In [1]:
# pip install "gymnasium[mujoco]" stable-baselines3 huggingface_sb3 "minari[all]"
import gymnasium as gym, minari
from stable_baselines3 import SAC
from huggingface_sb3 import load_from_hub

# The repo has two zips (hopper-v5-sac-expert.zip and hopper-v5-SAC-expert.zip).
# Try this one first; switch if it fails to load.
path = load_from_hub("farama-minari/Hopper-v5-SAC-expert", "hopper-v5-sac-expert.zip")
expert = SAC.load(path)

def expert_label(obs):                 # this is what DAgger queries
    return expert.predict(obs, deterministic=True)[0]

env = gym.make("Hopper-v5")
ds_expert = minari.load_dataset("mujoco/hopper/expert-v0", download=True)
ds_medium = minari.load_dataset("mujoco/hopper/medium-v0", download=True)
ds_simple = minari.load_dataset("mujoco/hopper/simple-v0", download=True)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/home/mahesh_tapeti/miniconda3/envs/temporal/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/mahesh_tapeti/miniconda3/envs/temporal/lib/python3.10/site-packages/huggingface_hub/file_download.py:1727: DeprecationWarning: hf_xet.download_files() is deprecated. Use XetSession().new_file_download_group().start_download_file() instead.
  xet_get(
/home/mahesh_tapeti/miniconda3/envs/temporal/lib/python3.10/site-packages/stable_baselines3/common/sa

Fetching 2 files:  50%|█████     | 1/2 [00:00<00:00,  3.76it/s]/home/mahesh_tapeti/miniconda3/envs/temporal/lib/python3.10/site-packages/huggingface_hub/file_download.py:1727: DeprecationWarning: hf_xet.download_files() is deprecated. Use XetSession().new_file_download_group().start_download_file() instead.
  xet_get(
Fetching 2 files: 100%|██████████| 2/2 [01:17<00:00, 38.52s/it]



Dataset mujoco/hopper/expert-v0 downloaded to /home/mahesh_tapeti/.minari/datasets/mujoco/hopper/expert-v0



Fetching 2 files: 100%|██████████| 2/2 [00:02<00:00,  1.24s/it]



Dataset mujoco/hopper/medium-v0 downloaded to /home/mahesh_tapeti/.minari/datasets/mujoco/hopper/medium-v0



Fetching 2 files: 100%|██████████| 2/2 [00:06<00:00,  3.04s/it]


Dataset mujoco/hopper/simple-v0 downloaded to /home/mahesh_tapeti/.minari/datasets/mujoco/hopper/simple-v0


In [16]:
ds = ds_medium
print("episodes:", ds.total_episodes, "| steps:", ds.total_steps)
print("obs space:", ds.observation_space)
print("act space:", ds.action_space)

episodes: 1327 | steps: 999404
obs space: Box(-inf, inf, (11,), float64)
act space: Box(-1.0, 1.0, (3,), float32)


In [17]:
datasets = {"expert": ds_expert, "medium": ds_medium, "simple": ds_simple}

for name, ds in datasets.items():
    print(f"\n=== {name} ===")
    print("episodes:", ds.total_episodes, "| steps:", ds.total_steps)
    print("obs space:", ds.observation_space)
    print("act space:", ds.action_space)


=== expert ===
episodes: 1086 | steps: 999164
obs space: Box(-inf, inf, (11,), float64)
act space: Box(-1.0, 1.0, (3,), float32)

=== medium ===
episodes: 1327 | steps: 999404
obs space: Box(-inf, inf, (11,), float64)
act space: Box(-1.0, 1.0, (3,), float32)

=== simple ===
episodes: 1952 | steps: 999206
obs space: Box(-inf, inf, (11,), float64)
act space: Box(-1.0, 1.0, (3,), float32)


In [5]:
import ocbench
from ocbench.oracles import BlockController
import random

env = ocbench.make("block-cpu-single-task1-v0")
expert = BlockController(env=env)

T = 100
D_new   = []
ob, info = env.reset()
expert.reset(ob, info, transition_idx=0)
for t in range(T):
    a_expert = expert.select_action(ob, info, transition_idx=t)   # label; advances the plan
    D_new.append((ob, a_expert))
    a_exec = a_expert if random() < beta else policy(ob)            # who drives
    ob, r, term, trunc, info = env.step(a_exec)
    # if the learner caused a big failure (dropped/knocked cube), force a replan:
    # if deviation_too_large(info): expert._reset_active(ob, info, t)

TypeError: 'module' object is not callable